In [ ]:

# ROUND ROBIN CPU SCHEDULING VISUALIZER
# Operating Systems Mini Project - Google Colab

import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# -------------------------------
# INPUT SECTION
# -------------------------------

process_count = widgets.IntSlider(
    value=4, min=1, max=10, step=1,
    description="Processes:",
    style={"description_width": "initial"}
)

time_quantum = widgets.IntSlider(
    value=2, min=1, max=10, step=1,
    description="Time Quantum:",
    style={"description_width": "initial"}
)

arrival_inputs = []
burst_inputs = []

for i in range(10):
    arrival_inputs.append(
        widgets.IntText(
            value=0, description=f"P{i+1} Arrival:",
            style={"description_width": "initial"},
            layout=widgets.Layout(width="220px")
        )
    )
    burst_inputs.append(
        widgets.IntText(
            value=[5, 4, 6, 3, 2, 4, 5, 3, 6, 2][i],
            description=f"P{i+1} Burst:",
            style={"description_width": "initial"},
            layout=widgets.Layout(width="200px")
        )
    )

input_box = widgets.VBox()
output = widgets.Output()

def update_inputs(change=None):
    rows = []
    for i in range(process_count.value):
        rows.append(
            widgets.HBox([
                arrival_inputs[i],
                burst_inputs[i]
            ])
        )
    input_box.children = tuple(rows)

process_count.observe(update_inputs, names="value")
update_inputs()

run_button = widgets.Button(
    description="Run Scheduling",
    button_style="success",
    icon="play"
)

reset_button = widgets.Button(
    description="Reset Values",
    button_style="warning",
    icon="refresh"
)

# -------------------------------
# ROUND ROBIN ALGORITHM
# -------------------------------

def run_scheduling(button):
    with output:
        clear_output(wait=True)

        n = process_count.value
        quantum = time_quantum.value

        if quantum <= 0:
            print("Time quantum must be greater than zero.")
            return

        processes = []

        for i in range(n):
            at = arrival_inputs[i].value
            bt = burst_inputs[i].value

            if at < 0 or bt <= 0:
                print(
                    f"Invalid values for P{i+1}. "
                    "Arrival time must be >= 0 and burst time > 0."
                )
                return

            processes.append({
                "Process": f"P{i+1}",
                "Arrival Time": at,
                "Burst Time": bt,
                "Remaining": bt,
                "Completion Time": 0,
                "Waiting Time": 0,
                "Turnaround Time": 0
            })

        # Sort by arrival time
        processes.sort(key=lambda p: p["Arrival Time"])

        ready_queue = []
        timeline = []
        time = 0
        index = 0
        completed = 0

        while completed < n:

            # Add processes that have arrived
            while (index < n and
                   processes[index]["Arrival Time"] <= time):
                ready_queue.append(processes[index])
                index += 1

            # CPU is idle if no process is ready
            if not ready_queue:
                next_arrival = processes[index]["Arrival Time"]
                if time < next_arrival:
                    timeline.append(
                        ("Idle", time, next_arrival)
                    )
                    time = next_arrival
                continue

            current = ready_queue.pop(0)

            start = time
            run_time = min(quantum, current["Remaining"])
            time += run_time
            current["Remaining"] -= run_time

            timeline.append((current["Process"], start, time))

            # Add new arrivals before re-queueing current process
            while (index < n and
                   processes[index]["Arrival Time"] <= time):
                ready_queue.append(processes[index])
                index += 1

            if current["Remaining"] > 0:
                ready_queue.append(current)
            else:
                current["Completion Time"] = time
                completed += 1

        # Calculate results
        for p in processes:
            p["Turnaround Time"] = (
                p["Completion Time"] - p["Arrival Time"]
            )
            p["Waiting Time"] = (
                p["Turnaround Time"] - p["Burst Time"]
            )

        # -------------------------------
        # RESULTS TABLE
        # -------------------------------

        df = pd.DataFrame(processes)
        df = df.drop(columns=["Remaining"])
        df = df.sort_values(
            by="Process",
            key=lambda s: s.str[1:].astype(int)
        ).reset_index(drop=True)

        print("ROUND ROBIN SCHEDULING RESULTS")
        print("-" * 50)
        display(df)

        avg_wait = df["Waiting Time"].mean()
        avg_turnaround = df["Turnaround Time"].mean()

        print(f"Average Waiting Time: {avg_wait:.2f}")
        print(f"Average Turnaround Time: {avg_turnaround:.2f}")

        # -------------------------------
        # GANTT CHART
        # -------------------------------

        fig, ax = plt.subplots(figsize=(12, 3))

        for process, start, end in timeline:
            ax.barh(
                0, end - start,
                left=start,
                height=0.5,
                edgecolor="black",
                label=process
            )
            ax.text(
                (start + end) / 2, 0,
                process,
                ha="center", va="center",
                fontsize=10
            )
            ax.text(
                start, -0.38,
                str(start),
                ha="center", va="top"
            )

        if timeline:
            ax.text(
                timeline[-1][2], -0.38,
                str(timeline[-1][2]),
                ha="center", va="top"
            )

        ax.set_yticks([])
        ax.set_xlabel("Time")
        ax.set_title("Round Robin Gantt Chart")
        ax.grid(axis="x", linestyle="--", alpha=0.4)
        ax.set_xlim(
            0,
            max(1, timeline[-1][2] if timeline else 1) + 1
        )
        plt.tight_layout()
        plt.show()

        print("\nExecution Order:")
        print(" → ".join(
            f"{p} ({s}-{e})" for p, s, e in timeline
        ))

# -------------------------------
# RESET BUTTON
# -------------------------------

def reset_values(button):
    process_count.value = 4
    time_quantum.value = 2

    default_burst = [5, 4, 6, 3, 2, 4, 5, 3, 6, 2]

    for i in range(10):
        arrival_inputs[i].value = 0
        burst_inputs[i].value = default_burst[i]

    with output:
        clear_output(wait=True)

run_button.on_click(run_scheduling)
reset_button.on_click(reset_values)

# -------------------------------
# DISPLAY INTERFACE
# -------------------------------

display(widgets.HTML(
    "<h2>🔄 Round Robin CPU Scheduling</h2>"
    "<p>Change the values below and click Run Scheduling.</p>"
))

display(process_count)
display(time_quantum)
display(input_box)
display(widgets.HBox([run_button, reset_button]))
display(output)


HTML(value='<h2>🔄 Round Robin CPU Scheduling</h2><p>Change the values below and click Run Scheduling.</p>')

IntSlider(value=4, description='Processes:', max=10, min=1, style=SliderStyle(description_width='initial'))

IntSlider(value=2, description='Time Quantum:', max=10, min=1, style=SliderStyle(description_width='initial'))

Output()

In [ ]:

%%html
<div id="rr-app">
<style>
#rr-app {
  font-family: Arial, sans-serif;
  color: #eaf3ff;
  background: #091c2b;
  padding: 22px;
  border-radius: 16px;
}
#rr-app * { box-sizing: border-box; }
#rr-app .grid {
  display: grid;
  grid-template-columns: 0.9fr 1.4fr;
  gap: 18px;
}
#rr-app .panel {
  background: rgba(17,39,55,.94);
  border: 1px solid #233f53;
  border-radius: 14px;
  padding: 22px;
  margin-bottom: 18px;
  box-shadow: 0 8px 22px #0002;
}
#rr-app h2, #rr-app h3 { margin-top: 0; }
#rr-app h2 { font-size: 23px; }
#rr-app h3 { font-size: 17px; color: #b8ccdc; }
#rr-app .muted { color: #9bb1c3; }
#rr-app .big { font-size: 27px; font-weight: bold; }
#rr-app .red { color: #ff5268; }
#rr-app .green { color: #36dfad; }
#rr-app label { display: block; margin: 12px 0 5px; color: #b8ccdc; }
#rr-app input {
  width: 100%; padding: 10px; border-radius: 7px;
  border: 1px solid #365369; background: #0b2232;
  color: white; font-size: 15px;
}
#rr-app button {
  border: 0; border-radius: 8px; padding: 11px 15px;
  margin: 8px 5px 0 0; cursor: pointer;
  font-weight: bold; color: #071c2b; background: #39d9b0;
}
#rr-app button.secondary { background: #294556; color: white; }
#rr-app button:disabled { opacity: .45; cursor: not-allowed; }
#rr-app .tablewrap { overflow-x: auto; }
#rr-app table { width: 100%; border-collapse: collapse; min-width: 470px; }
#rr-app th { color: #a9bfd0; font-size: 12px; text-align: left; }
#rr-app th, #rr-app td { padding: 12px 7px; border-bottom: 1px solid #294152; }
#rr-app .track { background: #243d4d; height: 12px; border-radius: 20px; overflow: hidden; }
#rr-app .bar { height: 100%; width: 0; background: #39d9b0; transition: width .2s; }
#rr-app .gantt { display: flex; gap: 4px; overflow-x: auto; padding: 15px 0; }
#rr-app .gblock {
  min-width: 65px; height: 48px; display: flex; align-items: center;
  justify-content: center; border-radius: 6px; color: #071c2b;
  font-weight: bold; position: relative;
}
#rr-app .gblock small { position: absolute; top: 51px; color: #a9bfd0; font-weight: normal; }
#rr-app .stats { display: grid; grid-template-columns: 1fr 1fr; gap: 12px; }
#rr-app .stat { background: #0b2232; padding: 15px; border-radius: 10px; }
#rr-app .stat strong { display: block; font-size: 24px; margin-top: 8px; }
#rr-app .controls { display: grid; grid-template-columns: 1fr 1fr; gap: 12px; }
#rr-app .process-input { display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 9px; }
#rr-app .notice { color: #ffc66d; font-size: 13px; margin-top: 12px; }
@media(max-width:750px) {
  #rr-app .grid { grid-template-columns: 1fr; }
  #rr-app { padding: 12px; }
}
</style>

<div class="grid">
  <div>
    <div class="panel">
      <div class="muted">ALGORITHM</div>
      <h2>Round Robin CPU Scheduling</h2>
      <div class="muted">CPU STATUS</div>
      <div class="big red" id="cpu">IDLE</div>
      <p class="muted">Timer</p>
      <div class="big" id="timer">0 s</div>
      <p class="muted">Time Quantum</p>
      <div class="big" id="quantumDisplay">2 s</div>
    </div>

    <div class="panel">
      <h3>Scheduling Settings</h3>
      <label>Number of processes (1–8)</label>
      <input id="count" type="number" min="1" max="8" value="4">
      <label>Time quantum</label>
      <input id="quantum" type="number" min="1" value="2">
      <button onclick="rrMakeInputs()" class="secondary">Update Processes</button>
      <div id="inputs"></div>
      <button onclick="rrStart()">▶ Run Scheduling</button>
      <button onclick="rrPause()" class="secondary" id="pauseBtn" disabled>Pause</button>
      <button onclick="rrReset()" class="secondary">Reset</button>
      <div class="notice">Enter arrival time and burst time for each process.</div>
      <div id="message" class="notice"></div>
    </div>

    <div class="panel">
      <h3>Performance</h3>
      <div class="stats">
        <div class="stat">Average Waiting Time<strong id="avgWT">0 s</strong></div>
        <div class="stat">Average Turnaround Time<strong id="avgTAT">0 s</strong></div>
      </div>
    </div>
  </div>

  <div>
    <div class="panel">
      <h2>Process Monitor</h2>
      <div class="tablewrap">
        <table>
          <thead>
            <tr>
              <th>PID</th><th>AT</th><th>STATUS BAR</th>
              <th>REMAINING BT</th><th>WT</th><th>CT</th>
            </tr>
          </thead>
          <tbody id="processRows"></tbody>
        </table>
      </div>
    </div>

    <div class="panel">
      <h3>CPU Execution Timeline</h3>
      <div class="muted">Gantt Chart</div>
      <div class="gantt" id="gantt"><span class="muted">Run scheduling to see the chart.</span></div>
    </div>

    <div class="panel">
      <h3>Execution Log</h3>
      <div id="log" class="muted">Waiting for scheduling to start...</div>
    </div>
  </div>
</div>
</div>

<script>
(() => {
  const root = document.getElementById('rr-app');
  const $ = id => root.querySelector('#' + id);
  let processes = [], timeline = [], timerId = null;
  let cursor = 0, paused = false, running = false;
  const colors = ['#39d9b0','#66b4ff','#ffc66d','#f68ad4',
                  '#b9a1ff','#f68c75','#91d9e8','#b5db77'];

  window.rrMakeInputs = function() {
    rrReset();
    let n = Number($('count').value);
    if (!Number.isInteger(n) || n < 1 || n > 8) {
      $('message').textContent = 'Choose between 1 and 8 processes.';
      return;
    }
    $('inputs').innerHTML = '';
    for (let i = 0; i < n; i++) {
      let row = document.createElement('div');
      row.innerHTML = `
        <div class="process-input">
          <div><label>Process</label><input disabled value="P${i+1}"></div>
          <div><label>Arrival time</label><input type="number" min="0" value="0" id="at${i}"></div>
          <div><label>Burst time</label><input type="number" min="1" value="${[5,4,6,3,2,4,5,3][i]}" id="bt${i}"></div>
        </div>`;
      $('inputs').appendChild(row);
    }
    $('message').textContent = 'Edit the values and click Run Scheduling.';
    rrRenderTable();
  };

  window.rrReset = function() {
    clearInterval(timerId);
    timerId = null; running = false; paused = false;
    cursor = 0; timeline = []; processes = [];
    $('timer').textContent = '0 s';
    $('cpu').textContent = 'IDLE';
    $('cpu').className = 'big red';
    $('quantumDisplay').textContent = ($('quantum').value || 2) + ' s';
    $('avgWT').textContent = '0 s';
    $('avgTAT').textContent = '0 s';
    $('gantt').innerHTML = '<span class="muted">Run scheduling to see the chart.</span>';
    $('log').textContent = 'Waiting for scheduling to start...';
    $('pauseBtn').disabled = true;
    $('pauseBtn').textContent = 'Pause';
    $('message').textContent = '';
    rrRenderTable();
  };

  function rrRenderTable() {
    $('processRows').innerHTML = processes.map(p => `
      <tr>
        <td>${p.pid}</td><td>${p.at} s</td>
        <td><div class="track"><div class="bar" style="width:${(p.bt-p.rem)/p.bt*100}%;background:${colors[p.id%8]}"></div></div></td>
        <td>${p.rem} s</td>
        <td>${p.done ? p.wt+' s' : '–'}</td>
        <td>${p.done ? p.ct+' s' : '–'}</td>
      </tr>`).join('');
  }

  function rrRenderGantt() {
    if (!timeline.length) return;
    $('gantt').innerHTML = timeline.map((t,i) => `
      <div class="gblock" style="background:${t.pid==='IDLE'?'#78909c':colors[t.id%8]};flex:${Math.max(1,t.end-t.start)}">
        ${t.pid}<small>${t.start}</small>
      </div>`).join('') + `<span class="muted">${timeline[timeline.length-1].end}</span>`;
  }

  window.rrStart = function() {
    rrReset();
    const n = Number($('count').value);
    const q = Number($('quantum').value);
    if (!Number.isInteger(n) || n < 1 || n > 8 ||
        !Number.isInteger(q) || q < 1) {
      $('message').textContent = 'Enter a valid process count and positive time quantum.';
      return;
    }
    for (let i=0;i<n;i++) {
      const at = Number($('at'+i)?.value);
      const bt = Number($('bt'+i)?.value);
      if (!Number.isInteger(at) || at < 0 ||
          !Number.isInteger(bt) || bt < 1) {
        $('message').textContent = `Invalid values for P${i+1}. Arrival must be >= 0 and burst must be > 0.`;
        return;
      }
      processes.push({id:i,pid:'P'+(i+1),at,bt,rem:bt,ct:0,wt:0,tat:0,done:false});
    }
    $('quantumDisplay').textContent = q+' s';

    // Calculate the complete Round Robin schedule first.
    let queue = [], sorted = [...processes].sort((a,b)=>a.at-b.at || a.id-b.id);
    let idx=0, time=0, finished=0, safety=0;
    while(finished<n && safety++<100000) {
      while(idx<n && sorted[idx].at<=time) queue.push(sorted[idx++]);
      if(!queue.length) {
        let next=sorted[idx].at;
        timeline.push({pid:'IDLE',id:0,start:time,end:next});
        time=next;
        continue;
      }
      let p=queue.shift(), start=time;
      let duration=Math.min(q,p.rem);
      time+=duration; p.rem-=duration;
      timeline.push({pid:p.pid,id:p.id,start,end:time});
      while(idx<n && sorted[idx].at<=time) queue.push(sorted[idx++]);
      if(p.rem>0) queue.push(p);
      else {p.done=true;p.ct=time;p.tat=p.ct-p.at;p.wt=p.tat-p.bt;finished++;}
    }
    if(finished<n) {
      $('message').textContent='Unable to calculate this schedule.';
      timeline=[]; return;
    }

    // Save final results, then animate the CPU timeline.
    const results=processes.map(p=>({...p}));
    const finalTime=timeline.length?timeline[timeline.length-1].end:0;
    const totalWT=results.reduce((s,p)=>s+p.wt,0)/n;
    const totalTAT=results.reduce((s,p)=>s+p.tat,0)/n;
    $('avgWT').textContent=totalWT.toFixed(2)+' s';
    $('avgTAT').textContent=totalTAT.toFixed(2)+' s';
    processes.forEach(p=>{p.rem=p.bt;p.done=false;p.ct=0;p.wt=0;p.tat=0;});
    cursor=0; running=true; paused=false;
    $('pauseBtn').disabled=false;
    $('message').textContent='Schedule calculated. Watch each CPU time slice.';
    rrRenderTable();
    rrRenderGantt();
    $('gantt').innerHTML='<span class="muted">Executing...</span>';
    $('log').textContent='Starting CPU...';
    timerId=setInterval(()=>{
      if(paused) return;
      if(cursor>=timeline.length) {
        clearInterval(timerId); timerId=null; running=false;
        $('cpu').textContent='COMPLETED'; $('cpu').className='big green';
        $('timer').textContent=finalTime+' s';
        $('gantt').innerHTML='';
        timeline.forEach(t=>{
          const block=document.createElement('div');
          block.className='gblock';
          block.style.flex=Math.max(1,t.end-t.start);
          block.style.background=t.pid==='IDLE'?'#78909c':colors[t.id%8];
          block.textContent=t.pid;
          const small=document.createElement('small');
          small.textContent=t.start;
          block.appendChild(small); $('gantt').appendChild(block);
        });
        const end=document.createElement('span');
        end.className='muted'; end.textContent=finalTime;
        $('gantt').appendChild(end);
        processes.forEach((p,i)=>{
          let r=results.find(x=>x.id===p.id);
          Object.assign(p,{rem:0,done:true,ct:r.ct,wt:r.wt,tat:r.tat});
        });
        rrRenderTable();
        $('log').textContent='All processes completed successfully.';
        $('pauseBtn').disabled=true;
        $('message').textContent='Scheduling completed!';
        return;
      }
      const t=timeline[cursor++];
      $('cpu').textContent=t.pid;
      $('cpu').className='big '+(t.pid==='IDLE'?'red':'green');
      $('timer').textContent=t.end+' s';
      $('log').textContent=`${t.pid}: executing from ${t.start} s to ${t.end} s`;
      processes.forEach(p=>{
        let used=0;
        timeline.slice(0,cursor).forEach(x=>{if(x.id===p.id && x.pid!=='IDLE')used+=x.end-x.start;});
        p.rem=p.bt-used;
        let r=results.find(x=>x.id===p.id);
        if(p.rem===0) Object.assign(p,{done:true,ct:r.ct,wt:r.wt,tat:r.tat});
      });
      rrRenderTable();
      $('gantt').innerHTML='';
      timeline.slice(0,cursor).forEach(x=>{
        const b=document.createElement('div'); b.className='gblock';
        b.style.flex=Math.max(1,x.end-x.start);
        b.style.background=x.pid==='IDLE'?'#78909c':colors[x.id%8];
        b.textContent=x.pid;
        const s=document.createElement('small'); s.textContent=x.start;
        b.appendChild(s); $('gantt').appendChild(b);
      });
      const e=document.createElement('span'); e.className='muted'; e.textContent=t.end;
      $('gantt').appendChild(e);
    },650);
  };

  window.rrPause = function() {
    if(!running) return;
    paused=!paused;
    $('pauseBtn').textContent=paused?'Resume':'Pause';
    $('cpu').textContent=paused?'PAUSED':$('cpu').textContent;
  };

  rrMakeInputs();
})();
</script>


PID,AT,STATUS BAR,REMAINING BT,WT,CT
